In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
from typing import List

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, precision_recall_curve, auc

from src.evaluation import evaluate_run


def minimum(dq_results: pd.DataFrame, certainties: pd.DataFrame) -> pd.Series:
    return dq_results.min(axis=0)


def maximum(dq_results: pd.DataFrame, certainties: pd.DataFrame) -> pd.Series:
    return dq_results.max(axis=0)


def weighted_minimum(dq_results: pd.DataFrame, certainties: pd.DataFrame) -> pd.Series:
    weighted_results = dq_results * certainties
    return weighted_results.min(axis=0)


def weighted_maximum(dq_results: pd.DataFrame, certainties: pd.DataFrame) -> pd.Series:
    weighted_results = dq_results * certainties
    return weighted_results.max(axis=0)


def mean(dq_results: pd.DataFrame, certainties: pd.DataFrame) -> pd.Series:
    return dq_results.mean(axis=0)


def weighted_mean(dq_results: pd.DataFrame, certainties: pd.DataFrame) -> pd.Series:
    weighted_sums = (dq_results * certainties).sum(axis=0)
    sum_of_weights = certainties.sum(axis=0)
    return weighted_sums / sum_of_weights


def median(dq_results: pd.DataFrame, certainties: pd.DataFrame) -> pd.Series:
    return dq_results.median(axis=0)


def weighted_median(dq_results: pd.DataFrame, certainties: pd.DataFrame) -> pd.Series:
    weighted_results = dq_results * certainties
    return weighted_results.median(axis=0)


def evaluate_aggregation_methods(
    dq_results: pd.DataFrame, certainties: pd.DataFrame, expected: pd.DataFrame
) -> pd.DataFrame:
    aggregation_methods = [
        minimum,
        maximum,
        weighted_minimum,
        weighted_maximum,
        mean,
        weighted_mean,
        median,
        weighted_median,
    ]

    mse_results = {}
    for method in aggregation_methods:
        aggregated_results = method(dq_results, certainties)
        mse = ((expected - aggregated_results) ** 2).mean()
        mse_results[method.__name__] = mse

    return pd.DataFrame(mse_results)


def plot_accuracy_by_threshold(
    dq_results: pd.DataFrame, mask: pd.DataFrame, columns: List[str]
):
    # Accuracy (correctly classified observations) should not be used on imbalanced data
    thresholds = [i / 100 for i in range(0, 101)]
    accuracies_per_column = {}
    for threshold in thresholds:
        accuracy = ((dq_results[columns] >= threshold) == (~mask[columns])).mean()
        for col in accuracy.index:
            accuracies_per_column.setdefault(col, []).append(accuracy[col])

    fig, ax = plt.subplots()

    for col, accuracies in accuracies_per_column.items():
        ax.plot(thresholds, accuracies, label=col)
    ax.set_xlabel("DQ threshold")
    ax.set_ylabel("Accuracy")
    ax.set_title("DQ Accuracy by Threshold")
    ax.grid()
    ax.legend()
    plt.show(fig)


def plot_f1_score_by_threshold(
    dq_results: pd.DataFrame, mask: pd.DataFrame, columns: List[str]
):
    thresholds = [i / 100 for i in range(0, 101)]
    f1_scores_per_column = {}
    for threshold in thresholds:
        predictions = dq_results[columns] >= threshold
        for col in columns:
            f1 = f1_score(~mask[col], predictions[col])
            f1_scores_per_column.setdefault(col, []).append(f1)

    fig, ax = plt.subplots()

    for col, f1_scores in f1_scores_per_column.items():
        ax.plot(thresholds, f1_scores, label=col)
    ax.set_xlabel("DQ threshold")
    ax.set_ylabel("F1 Score")
    ax.set_title("DQ F1 Score by Threshold")
    ax.grid()
    ax.legend()
    plt.show(fig)


def plot_pr_auc_curve(dq_results: pd.DataFrame, mask: pd.DataFrame, columns: List[str]):
    fig, ax = plt.subplots()

    for col in columns:
        precision, recall, t = precision_recall_curve(
            ~mask[col][dq_results[col].dropna().index], dq_results[col].dropna()
        )
        pr_auc = auc(recall, precision)
        ax.plot(recall, precision, label=f"{col} (AUC={pr_auc:.2f})")

    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title("Precision-Recall Curve")
    ax.grid()
    ax.legend()
    plt.show(fig)


with pd.option_context("display.expand_frame_repr", False):
    # Completeness
    # evaluate_run("20260131_165047")
    # Consistency
    # evaluate_run("20260201_220316")
    # Timeliness
    # evaluate_run("20260201_223309") # Auto Sales
    evaluate_run("20260202_102230")  # Open Library

None

/var/folders/rw/xym_kq6n3l9_0qkvb5l3bd_w0000gn/T/ipykernel_16841/3149088382.py:92: DtypeWarning: Columns (0: contributions, 1: edition_name, 2: genres, 3: publish_country, 4: by_statement, 5: other_titles, 6: publish_places, 7: dewey_decimal_class, 8: lccn, 9: title_prefix, 10: url, 11: work_titles, 12: table_of_contents, 13: oclc_number, 14: description, 15: subject_place, 16: work_title, 17: subject_time, 18: isbn_invalid, 19: weight, 20: physical_dimensions, 21: first_sentence, 22: ocaid, 23: location, 24: isbn_odd_length, 25: openlibrary, 26: coverimage, 27: uri_descriptions, 28: uris, 29: purchase_url, 30: language, 31: contributors, 32: full_title, 33: scan_on_demand, 34: isbn, 35: collections, 36: copyright_date, 37: local_id, 38: create, 39: volumes, 40: scan_records, 41: ia_box_id, 42: ia_loaded_id, 43: links, 44: subject_places, 45: subject_times, 46: subject_people, 47: translated_from, 48: translation_of, 49: providers, 50: lc_classification, 51: oclc, 52: author) have mixe

No mask found for open_library
Saved evaluations to /Users/jberndt/Documents/Masterarbeit/notebooks/results/20260202_102230/evaluations.json
